# 2 — Fetch ATM Strike OHLC Data (Continuous Loop)

**Run frequency:** Every trading day, during market hours (09:15 – 15:30 IST)

**What this does:**
1. Logs in to Zerodha KiteConnect via automated TOTP
2. Starts a SparkSession
3. Reads instrument data from Parquet (written by Notebook 1)
4. Determines the ATM strike for BankNifty (live LTP or custom override)
5. Selects N strikes above/below ATM for the current expiry
6. Fetches 3-minute OHLC + OI history for each strike via KiteConnect API
7. Writes data to partitioned Parquet files
8. **Loops every minute** to keep pulling and appending the latest candle

**Runs in parallel with Notebook 3** (charts will auto-refresh as new data arrives)

---
### ⚙️ Configuration
Edit the `CONFIG` cell below to adjust parameters.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `BANKNIFTY` | `True` | Pull BankNifty data |
| `NIFTY` | `False` | Pull Nifty data |
| `CUSTOM_STRIKE` | `0` | Override ATM strike (0 = live LTP) |
| `NUM_STRIKES` | `9` | N strikes each side of ATM |
| `PULL_NEXT_EXPIRY` | `False` | Also pull next weekly expiry |
| `NUM_DAYS_HISTORY` | `1` | Days of history per pull |
| `LOOP_INTERVAL_MIN` | `1` | Minutes between pulls |

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')  # Run from repo root so DataFiles/ paths resolve correctly

In [2]:
# ── CONFIG — edit these values ─────────────────────────────────────────────
BANKNIFTY         = True     # Pull BankNifty options data
NIFTY             = False    # Pull Nifty options data
CUSTOM_STRIKE     = 0        # 0 = live LTP; set e.g. 56500 to override ATM
NUM_STRIKES       = 9        # How many ITM/OTM strikes each side of ATM
PULL_NEXT_EXPIRY  = False    # Also pull next weekly expiry
NUM_DAYS_HISTORY  = 1        # Calendar days of history per pull
LOOP_INTERVAL_MIN = 1        # Loop frequency in minutes
INTERVAL          = '3minute'

In [3]:
# ── Step 1: Login ──────────────────────────────────────────────────────────
from utils.kite_helpers import kite_login, get_spark_session

kite, kws, access_token = kite_login()
print('✅ Login successful')

2026-03-10 23:36:19,135 [INFO] utils.kite_helpers — Logging in as user: ZB4988
2026-03-10 23:36:19,272 [INFO] utils.kite_helpers — Generated TOTP pin for 2FA
2026-03-10 23:36:19,430 [INFO] utils.kite_helpers — Login successful — request_token: 6yaaMCHCN9uXClPPA2bfg7JgBJJNAT43
2026-03-10 23:36:19,638 [INFO] utils.kite_helpers — KiteConnect session established for user: ZB4988


✅ Login successful


In [4]:
# ── Step 2: Start SparkSession ─────────────────────────────────────────────
spark = get_spark_session(app_name='2_Fetch_Strikes_Data')
print('✅ Spark session ready')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 23:36:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark session ready


In [5]:
# ── Step 3: Run continuous data pull loop ──────────────────────────────────
# This cell blocks until you interrupt the kernel (Kernel → Interrupt)
from utils.data_fetcher import run_data_loop

run_data_loop(
    kite                  = kite,
    spark                 = spark,
    banknifty             = BANKNIFTY,
    nifty                 = NIFTY,
    custom_strike         = CUSTOM_STRIKE,
    num_strikes           = NUM_STRIKES,
    pull_next_expiry      = PULL_NEXT_EXPIRY,
    num_days_history      = NUM_DAYS_HISTORY,
    interval              = INTERVAL,
    loop_interval_minutes = LOOP_INTERVAL_MIN,
)

2026-03-10 23:39:34,257 [ERROR] utils.data_fetcher — Data pull failed: [PATH_NOT_FOUND] Path does not exist: file:/Users/yogeshpatil/Library/CloudStorage/OneDrive-Personal/Trading/GitHub/CHEWY_BNF_Options_Trading/DataFiles/Instruments/Banknifty_Options.
Traceback (most recent call last):
  File "/Users/yogeshpatil/Library/CloudStorage/OneDrive-Personal/Trading/GitHub/CHEWY_BNF_Options_Trading/utils/data_fetcher.py", line 326, in _job
    get_latest_data(
  File "/Users/yogeshpatil/Library/CloudStorage/OneDrive-Personal/Trading/GitHub/CHEWY_BNF_Options_Trading/utils/data_fetcher.py", line 207, in get_latest_data
    ) = read_instruments(spark, instruments_base_path)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yogeshpatil/Library/CloudStorage/OneDrive-Personal/Trading/GitHub/CHEWY_BNF_Options_Trading/utils/strike_utils.py", line 295, in read_instruments
    bnf_options = spark.read.parquet(
                  ^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python

Running data pull at 2026-03-10 23:39:34
⚠️  Data pull error: [PATH_NOT_FOUND] Path does not exist: file:/Users/yogeshpatil/Library/CloudStorage/OneDrive-Personal/Trading/GitHub/CHEWY_BNF_Options_Trading/DataFiles/Instruments/Banknifty_Options.
